                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |	without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Folder of this repository (it holds DDC_WINDOWS_Converters and ddc-tools)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import convert

# Determining the path where the Revit / IFC files are located and creating new folders
path = os.environ.get("DDC_BATCH", r'C:\ProjectRevitIFC')
result = os.path.join(path, 'quick_rough_estimate')            # the quick QTO files
tables = os.path.join(result, 'tables')                        # flat tables (CSV) of every model
os.makedirs(tables, exist_ok=True)

# Convert every model of the folder and write its flat table as CSV (the same as:  ddc convert <file> --csv <table>)
for file in os.listdir(path):
    stem, ext = os.path.splitext(file)
    if ext.lower() in ('.rvt', '.ifc'):
        csv_file = os.path.join(tables, stem + '.csv')
        if not os.path.exists(csv_file):
            convert.convert(os.path.join(path, file), work=os.path.join(tables, stem + '.converted'),
                            xlsx=os.path.join(tables, stem + '.xlsx'), csv=csv_file)

In [ ]:
# Properties for which we want to collect data on the amount of volume
propstr = ['Area', 'Volume', 'Width', 'Length', 'Perimeter', 'öööasdöööfake']

# The type of property by which the grouping will be carried out
search_parameter = 'Type Name'

# Main function for grouping data and saving a file
def crtable(filename):
    df = pd.read_csv(os.path.join(tables, filename), encoding='utf-8-sig', low_memory=False)

    #  Only the quantity columns that exist in this model, as numbers (empty or text values count as 0)
    propindf = [el for el in propstr if el in df.columns]
    if not propindf or search_parameter not in df.columns:
        print(f'Skipped {filename}: no quantity columns or no {search_parameter}')
        return
    for el in propindf:
        df[el] = pd.to_numeric(df[el], errors='coerce').fillna(0)

    # Summation of all data that are grouped by search_parameter located in the propindf columns
    df1 = pd.pivot_table(df, index=[search_parameter], values=propindf, aggfunc='sum')
    df1 = df1.add_prefix('Sum of ')

    # Determination of the number of elements in groups
    df2 = df.groupby([search_parameter])[propindf[0]].agg(['count'])
    dfallpar = pd.DataFrame()
    df['ID'] = df['ID'].astype(str)
    comma = lambda x: ', '.join(x.unique())
    df3 = df.groupby([search_parameter]).agg({'ID': comma})

    # Collecting data into one dataframe
    dfallpar = pd.concat([df2, df1, df3], axis=1)
    dfallpar.rename(columns=({'ID': 'Id´s', 'count': 'Amount'}), inplace=True,)

    # Saving data to file
    dfallpar.to_excel(os.path.join(result, 'QQTO_' + filename + '.xlsx'), sheet_name='Quick QTO')
    print('Saved QQTO_' + filename + '.xlsx:', len(dfallpar), 'groups')

for filename in os.listdir(tables):
    if filename.endswith("csv"):
        crtable(filename)